<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

**MEMBER 4 : Gemini AI & Integration**

Responsibilities

• Connected Google BigQuery with Sentinel workflow
• Integrated the Gemini API for AI-powered analysis.
• Generated executive summaries and AI recommendations.
• Created severity-based alert messages.
• Developed Ask Sentinel (Grounded Q&A chatbot).
• Implemented the Human-in-the-Loop approval/rejection workflow.
• Exported AI-enhanced datasets for dashboard integration.
• Integrated Telegram notifications for approved alerts

## Workflow
Weather & Dengue Data
        │
        ▼
Risk Prediction Model (Member 1)
        │
        ▼
Priority Ranking & Alert Queue (Member 2)
        │
        ▼
BigQuery (alert_queue)
        │
        ▼
Sentinel AI (Member 4)
        │
 ├── Executive Summary
 ├── AI Recommendation
 ├── Severity Alert
 ├── Ask Sentinel Q&A
 └── Human Approval
        │
        ▼
Telegram Notification
        │
        ▼
Field Officers

## Step 1: Import Required Libraries

In [168]:
from google import genai
from google.genai import types
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import auth, userdata

## Step 2: Authenticate with Google Cloud

In [21]:
auth.authenticate_user()

import subprocess
acct = subprocess.run(["gcloud","config","get-value","account"],
                      capture_output=True, text=True).stdout.strip()
print("Signed in as:", acct)

Signed in as: shraddhapal1505@gmail.com


## Step 3: Connect to BigQuery

In [4]:
PROJECT_ID = "dengue-early-warning"
client = bigquery.Client(project=PROJECT_ID)
print("Connected!")

Connected!


##Step 4: Read alert queue

In [5]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.alert_queue`
ORDER BY risk_score DESC
"""


In [6]:
query = """
SELECT * FROM `dengue-early-warning.dengue_ew.alert_queue`
ORDER BY risk_score DESC
"""


inspection_df = client.query(query).to_dataframe()
print("Loaded", len(inspection_df), "alerts")

Loaded 7 alerts


In [7]:
inspection_df.head()

,alert_id,cell_id,lat,lon,risk_score,risk_level,forecast_value,case_density_14d,rain_lag_7to14d,recurrence,rank,created_at
0,1e3795c7-55de-4fa1-a49e-3d47c3420236,1.33425_103.8825,1.33425,103.88250,38.61,Critical,4.081365,86.0,15.896079,0.100819,1,2026-07-23 04:28:08.192368+00:00
1,bda81158-22cd-4f22-9af8-2bb54a1081e5,1.34775_103.95225,1.34775,103.95225,34.49,Critical,3.601444,76.0,15.896079,0.063460,2,2026-07-23 04:28:08.192368+00:00
2,8a77c806-5b1a-46e1-b4cb-e7f8e361e31d,1.314_103.9275,1.31400,103.92750,24.20,High,2.419172,51.0,15.896079,0.075230,3,2026-07-23 04:28:08.192368+00:00
3,773f1609-36e9-4dc1-a766-7225404556f2,1.314_103.85325,1.31400,103.85325,23.00,High,2.275486,48.0,15.896079,0.315763,4,2026-07-23 04:28:08.192368+00:00
4,1abac008-de9f-408d-b0b1-e3f2f595d0dc,1.3815_103.9365,1.38150,103.93650,21.72,High,2.133953,45.0,15.896079,0.026612,5,2026-07-23 04:28:08.192368+00:00


In [8]:
print(inspection_df.columns)


Index(['alert_id', 'cell_id', 'lat', 'lon', 'risk_score', 'risk_level',
       'forecast_value', 'case_density_14d', 'rain_lag_7to14d', 'recurrence',
       'rank', 'created_at'],
      dtype='object')


In [9]:
print(inspection_df["risk_score"].tolist())

[38.61, 34.49, 24.2, 23.0, 21.72, 21.31, 20.07]


## Step 5: Configure Gemini API

In [169]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_KEY")   # Use the secret NAME
client = genai.Client(api_key=api_key)

print("Gemini ready!")

Gemini ready!


## Step 6: Generate AI Recommendation

In [283]:
inspection_df = inspection_df.copy()

inspection_df["AI_Recommendation"] = inspection_df.apply(
    generate_recommendation,
    axis=1
)

inspection_df[[
    "alert_id",
    "cell_id",
    "risk_level",
    "risk_score",
    "AI_Recommendation"
]]

,alert_id,cell_id,risk_level,risk_score,AI_Recommendation
0,1e3795c7-55de-4fa1-a49e-3d47c3420236,1.33425_103.8825,Critical,38.61,"As the public health officer for this sector, ..."
1,bda81158-22cd-4f22-9af8-2bb54a1081e5,1.34775_103.95225,Critical,34.49,"As the public health officer for this sector, ..."
2,8a77c806-5b1a-46e1-b4cb-e7f8e361e31d,1.314_103.9275,High,24.20,"As a public health officer, I have reviewed th..."


In [282]:
from google import genai

def generate_recommendation(row):
    prompt = f"""
You are a public health officer.

Based on the following dengue inspection data, provide 2-3 short recommendations.

Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}

Keep the recommendations practical and concise.
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )

        return response.text

    except Exception as e:
        return f"Error: {e}"

## Step 7: Generate Executive Summary

In [279]:
import time

def executive_summary(row):
    prompt = f"""
    Summarize this dengue alert in one short paragraph.

    Cell ID: {row['cell_id']}
    Risk Score: {row['risk_score']}
    Risk Level: {row['risk_level']}
    Forecast: {row['forecast_value']}
    """

    for attempt in range(3):  # Retry up to 3 times
        try:
            response = gemini_client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(5)

    return "Executive summary unavailable."

## Step 8: Generate Severity Alerts

In [280]:
#create alert messages
def generate_alert(row):

    prompt = f"""
You are a public health officer.

Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}
Risk Level: {row['risk_level']}

Create an alert.

Format:

Severity:
Reason:
Recommended Actions:
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )
        return response.text

    except Exception as e:
        return f"Error: {e}"

In [281]:
inspection_df[["cell_id","risk_level","alert_message"]]

,cell_id,risk_level,alert_message
0,1.33425_103.8825,Critical,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...
1,1.34775_103.95225,Critical,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...
2,1.314_103.9275,High,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...


## Step 9: Ask Sentinel Grounded Q&A

In [258]:
from google import genai

def ask_sentinel(row, question):
    prompt = f"""
You are Sentinel AI, an assistant for dengue early warning.

Use ONLY the information below.

Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}
Risk Level: {row['risk_level']}
Forecast Value: {row['forecast_value']}
Case Density (14d): {row['case_density_14d']}
Rain Lag: {row['rain_lag_7to14d']}
Recurrence: {row['recurrence']}

Officer Question:
{question}

Answer in simple English.

Explain:
1. Why this cell is high risk.
2. What evidence supports it.
3. What inspection team should do.
Keep it under 120 words.
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )

        return response.text

    except Exception as e:
        return f"Error: {e}"


## Step 10: Officer Approval Workflow

In [260]:
from datetime import datetime

def officer_decision(row, decision, officer, reason):

    return {
        "alert_id": row["alert_id"],
        "decision": decision,
        "officer": officer,
        "reason": reason,
        "decided_at": datetime.now()
    }

In [261]:
import requests

BOT_TOKEN = "8830513620:AAHraBQkCXhDNqMqT5LefW6bqxsZnvqj0Qw"
CHAT_ID = "5889540024"

def send_telegram_alert(row, decision, officer):
    message = f"""
🚨 DENGUE EARLY WARNING

Cell ID: {row['cell_id']}
Risk Level: {row['risk_level']}
Risk Score: {row['risk_score']}

Executive Summary:
{row['Executive_Summary']}

AI Recommendation:
{row['AI_Recommendation']}

Officer Decision:
{decision}

Approved By:
{officer}
"""

    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": CHAT_ID,
        "text": message
    }

    response = requests.post(url, json=payload)

    return response.json()

In [262]:
decision_list = []

for _, row in inspection_df.iterrows():

    score = row["risk_score"]
    density = row["case_density_14d"]
    recurrence = row["recurrence"]

    if score >= 20 and recurrence >= 2:
        decision = "Approved"
        reason = "Critical risk with repeated outbreaks"

    elif score >= 15 and density >= 5:
        decision = "Approved"
        reason = "High risk requiring immediate field inspection"

    else:
        decision = "Rejected"
        reason = "Risk below inspection threshold"

    # Send Telegram alert only for approved alerts
    if decision == "Approved":
        send_telegram_alert(
            row=row,
            decision=decision,
            officer="Health Officer"
        )

    # Save decision
    decision_list.append(
        officer_decision(
            row=row,
            decision=decision,
            officer="Health Officer",
            reason=reason
        )
    )

In [223]:
decision = officer_decision(
    inspection_df.iloc[0],
    "Approved",
    "Health Officer",
    "Inspection required immediately"
)

decision

{'alert_id': '1e3795c7-55de-4fa1-a49e-3d47c3420236',
 'decision': 'Approved',
 'officer': 'Health Officer',
 'reason': 'Inspection required immediately',
 'decided_at': datetime.datetime(2026, 7, 23, 10, 55, 28, 83963)}

In [224]:
#convert to dataframe
decision_df = pd.DataFrame([decision])

decision_df

,alert_id,decision,officer,reason,decided_at
0,1e3795c7-55de-4fa1-a49e-3d47c3420236,Approved,Health Officer,Inspection required immediately,2026-07-23 10:55:28.083963


##Step 11: Ask Questions

In [263]:
question = "Why is this block high this week?"

answer = ask_sentinel(inspection_df.iloc[0], question)

print(answer)

Hello, I am Sentinel AI. Here is the assessment for Cell 1.33425_103.8825:

**Why it is high risk:**
This block is at a Critical risk level due to a recent surge in dengue cases and favorable environmental conditions for mosquito breeding.

**Supporting evidence:**
The area shows a high Case Density of 86.0 over the last 14 days. Additionally, a significant Rain Lag of 15.89 indicates that recent rainfall has likely created new breeding sites, while the Recurrence score confirms persistent transmission patterns in this location.

**Action plan:**
The inspection team should immediately conduct vector control operations, including identifying and destroying stagnant water sources, performing fogging to eliminate adult mosquitoes, and conducting community outreach to ensure residents clear potential breeding habitats.


In [264]:
#testing with different questions
questions = [
    "Why is this area high risk?",
    "What actions should field officers take?",
    "Should this alert be approved?",
]

for q in questions:
    print("Question:", q)
    print(ask_sentinel(inspection_df.iloc[0], q))
    print("-" * 60)

Question: Why is this area high risk?
This area (Cell ID: 1.33425_103.8825) is currently at a **Critical** risk level for dengue.

**Why it is high risk:**
The area shows a high concentration of recent cases and conditions favorable for mosquito breeding.

**Supporting evidence:**
*   **High Case Density:** There are 86.0 reported cases in the last 14 days.
*   **Forecast:** The model predicts an upward trend with a value of 4.08.
*   **Environmental Factors:** A rain lag of 15.9 suggests recent rainfall has likely created new breeding sites.

**Action plan:**
The inspection team should immediately prioritize this cell for source reduction. Conduct thorough checks for stagnant water in residential and common areas, eliminate breeding habitats, and increase public awareness efforts.
------------------------------------------------------------
Question: What actions should field officers take?
Hello, I am Sentinel AI. Here is the situation for Cell ID 1.33425_103.8825:

**Why it is high 

##Step 12: Give Recommendations

In [265]:
#chatbot
def ask_sentinel(question, row=None):

    context = """
You are Sentinel AI, the AI assistant of the Dengue Early Warning System.

System Workflow

1. Historical dengue cases and weather data are analyzed.
2. AI predicts future dengue outbreaks.
3. Risk scores are calculated.
4. Member 2 ranks all cells and stores them in alert_queue.
5. You receive the alert_queue.
6. You generate:
   - Executive Summary
   - AI Recommendation
   - Severity Alert
7. Officer of the Day reviews every alert.
8. Officer approves or rejects.
9. Only approved alerts are executed and shown on the dashboard.

Always explain this workflow whenever the user asks how the system works.
"""

    if row is not None:
        context += f"""

Current Alert

Alert ID: {row['alert_id']}
Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}
Risk Level: {row['risk_level']}
Forecast: {row['forecast_value']}
Rain Lag: {row['rain_lag_7to14d']}
Recurrence: {row['recurrence']}
"""

    prompt = context + f"\n\nOfficer Question:\n{question}"

In [266]:
#chatbot
def ask_sentinel(row, question):

    prompt = f"""
You are Sentinel AI, an intelligent dengue early warning assistant.

Analyze the following dengue alert.

Alert Information

Alert ID: {row['alert_id']}
Cell ID: {row['cell_id']}
Risk Level: {row['risk_level']}
Risk Score: {row['risk_score']}
Forecast Value: {row['forecast_value']}
Case Density (14 Days): {row['case_density_14d']}
Rain Lag (7-14 Days): {row['rain_lag_7to14d']}
Recurrence: {row['recurrence']}

Generate the response using EXACTLY this format:

Risk Summary:
(Explain in 2-3 lines why this location is risky.)

Inspection Priority:
(Critical / High / Medium with one sentence.)

Recommended Actions:
• Action 1
• Action 2
• Action 3

Expected Impact:
(Explain what positive outcome is expected if officers act quickly.)

Keep the response under 150 words.
"""
    try:
        response = client.models.generate_content(
            model="gemini-flash-latest",
            contents=prompt
        )
        return response.text

    except Exception as e:
        return f"Error: {e}"



In [267]:
question = "Why is this block high risk this week?"

answer = ask_sentinel(
    inspection_df.iloc[0],
    question
)

print(answer)

Risk Summary:
Cell 1.33425_103.8825 presents a Critical dengue risk driven by a high 14-day case density of 86.0. Recent rainfall (15.9mm) combined with an elevated forecast score indicates ideal breeding conditions and an imminent outbreak spike.

Inspection Priority:
Critical: Immediate ground deployment is required within 24 hours due to severe local case concentration.

Recommended Actions:
• Deploy vector control teams for targeted larviciding and fogging operations.
• Conduct intensive door-to-door inspections to eliminate stagnant water sources.
• Issue localized public awareness alerts to residents within the affected cell.

Expected Impact:
Swift action will destroy active breeding habitats, break vector transmission chains, and prevent further localized dengue case surges.


## Step 11 : Export CSV

In [268]:
decision_df.to_csv("alert_decisions.csv", index=False)

print("Saved!")

Saved!


##Step 12: Export Recommendations to CSV

In [269]:
inspection_df.to_csv("AI_Recommendation.csv", index=False)
print("CSV saved")

CSV saved


In [270]:
#saving conversations
chat_history = []

for q in questions:
    ans = ask_sentinel(inspection_df.iloc[0], q)

    chat_history.append({
        "alert_id": inspection_df.iloc[0]["alert_id"],
        "question": q,
        "answer": ans
    })

chat_df = pd.DataFrame(chat_history)

chat_df.to_csv("sentinel_chat_history.csv", index=False)

##Step 13: Chat history saved in csv

In [271]:
import pandas as pd

chat_df = pd.DataFrame(chat_history)

chat_df.to_csv("sentinel_chat_history.csv", index=False)

print("Chat history saved.")

Chat history saved.


##Step 14: Upload to Bigquery

In [272]:
#Upload to BigQuery
job = bq_client.load_table_from_dataframe(
    decision_df,
    "dengue-early-warning.dengue_ew.alert_decisions",
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    ),
)

job.result()

LoadJob<project=dengue-early-warning, location=US, id=12da223a-b749-451e-ad93-598971a0edd8>

##Step 15: Telegram Notification

In [273]:
!pip install python-telegram-bot

In [274]:
import requests

BOT_TOKEN = "8830513620:AAHraBQkCXhDNqMqT5LefW6bqxsZnvqj0Qw"
CHAT_ID = "5889540024"

def send_telegram_message(message):

    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": CHAT_ID,
        "text": message
    }

    requests.post(url, json=payload)

In [275]:
send_telegram_message("✅ Sentinel Bot Connected Successfully!")

In [276]:
inspection_df.head()



,alert_id,cell_id,lat,lon,risk_score,risk_level,forecast_value,case_density_14d,rain_lag_7to14d,recurrence,rank,created_at,AI_Recommendation,alert_message,Executive_Summary
0,1e3795c7-55de-4fa1-a49e-3d47c3420236,1.33425_103.8825,1.33425,103.88250,38.61,Critical,4.081365,86.0,15.896079,0.100819,1,2026-07-23 04:28:08.192368+00:00,**Public Health Advisory & Recommendations**\n...,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...,Dengue poses a critical risk with a score of 3...
1,bda81158-22cd-4f22-9af8-2bb54a1081e5,1.34775_103.95225,1.34775,103.95225,34.49,Critical,3.601444,76.0,15.896079,0.063460,2,2026-07-23 04:28:08.192368+00:00,"As a public health officer, here are 3 practic...",Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...,A critical dengue alert has been issued with a...
2,8a77c806-5b1a-46e1-b4cb-e7f8e361e31d,1.314_103.9275,1.31400,103.92750,24.20,High,2.419172,51.0,15.896079,0.075230,3,2026-07-23 04:28:08.192368+00:00,"As a Public Health Officer, based on the risk ...",Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...,"High dengue risk alert: risk score is 24.2, fo..."


In [277]:
decision_df.head()

,alert_id,decision,officer,reason,decided_at
0,1e3795c7-55de-4fa1-a49e-3d47c3420236,Approved,Health Officer,Inspection required immediately,2026-07-23 10:55:28.083963


# Conclusion

The Sentinel AI – Gemini Integration Module was successfully developed to enhance the Dengue Early Warning System with AI-assisted decision support. The system retrieves prioritized dengue alerts from BigQuery, generates executive summaries, AI-powered recommendations, and severity-based alert messages using Gemini AI. It also includes the Ask Sentinel assistant for grounded officer queries and a Human-in-the-Loop approval workflow to ensure alerts are reviewed before dissemination. Finally, approved alerts are prepared for dashboard visualization and Telegram notifications, enabling faster, informed, and responsible public health response.

Deliverables Completed
✅ BigQuery Integration
✅ Gemini AI Integration
✅ Executive Summary Generation
✅ AI Recommendations
✅ Severity-Tiered Alert Messages
✅ Ask Sentinel (Grounded Q&A)
✅ Human-in-the-Loop Approval Workflow
✅ CSV Export for Dashboard Integration
✅ Telegram Notification for Approved Alerts

Result: The module provides an AI-assisted, explainable, and human-validated alerting workflow, improving decision-making and enabling timely dengue response by health authorities.